# ACS3930 Assignment 3

In this assignment, you will implement the algorithms we learned in the class to solve Sudoku CSP problem. 

Have fun with your Assignment 3!

Your name:

Your Student ID:

In [45]:
from random import *
from copy import deepcopy
from collections import defaultdict
from time import time
from sys import stdout

## Class Definition: CSP

The `CSP` class is designed to represent a constraint satisfaction problem. It is initialized with three key attributes:

`variables`: A list of variables involved in the CSP.

`adjList`: An adjacency list representing the constraints between variables; essentially, it maps each variable to a list of variables that have constraints with it.

`domains`: A dictionary mapping each variable to its domain, which is the set of values that the variable can take.


The `restore_domains` method undoes a supposition and all inferences from it. This is used when backtracking in the CSP solution process. It takes a dictionary of removals where keys are variables and values are the set of values that were removed from these variables' domains. It then restores these values to the domains.


The `conflicted_vars` method returns a list of variables that are in conflict in the current assignment. It iterates through all variables and includes those for which `nconflicts` with their currently assigned value and the current assignment is greater than 0, indicating at least one conflict.

In [2]:
class CSP(object):
    def __init__(self, variables = [], adjList = {}, domains = {}):
        self.variables = variables
        self.adjList = adjList
        self.domains = domains

    def restore_domains(self, removals):
        """ Undo a supposition and all inferences from it """
        for X in removals:
            self.domains[X] |= removals[X]

    # The following methods are used in min_conflict algorithm
    def nconflicts(self, X1, x, assignment):
        """
        Return the number of conflicts X1 = x has with other variables
        Subclasses may implement this more efficiently
        """
        def conflict(X2):
            return self.conflicts(X1, x, X2, assignment[X2])
        return sum(conflict(X2) for X2 in self.adjList[X1] if X2 in assignment)

    def conflicted_vars(self, current):
        """ Return a list of variables in conflict in current assignment """
        return [var for var in self.variables
                if self.nconflicts(var, current[var], current) > 0]

## Class Definition: SudokuCSP

Inherits from the `CSP` class to leverage the generic CSP infrastructure such as variables, domains, adjacency lists, and basic CSP operations. It is specialized to handle the specific constraint checking needed for Sudoku puzzles.

The `conflicts` method takes six parameters: `(i1, j1, x, i2, j2, y)`, where `i1, j1` and `i2, j2` are the row and column indices of two cells in the Sudoku grid, and `x` and `y` are the values assigned to these cells, respectively. The method calculates `k1` and `k2`, which are the indices of the 3x3 subgrids (also known as boxes or blocks) that contain the cells `(i1, j1)` and `(i2, j2)`. This calculation divides the grid into 3x3 sections and assigns a unique index to each.

It then returns `True` if the assigned values `x` and `y` are equal and at least one of the following conditions is met:
`i1 == i2`: The two cells are in the same row.
`j1 == j2`: The two cells are in the same column.
`k1 == k2`: The two cells are in the same 3x3 subgrid.

If any of these conditions is true, it means that the assignment of `x` to `(i1, j1)` and `y` to `(i2, j2)` violates the Sudoku rules, which state that each number 1-9 must appear exactly once in each row, each column, and each of the nine 3x3 subgrids.


This customized `conflicts` method allows the `SudokuCSP` class to directly support the validation of Sudoku constraints. It effectively checks whether two assignments conflict based on the Sudoku rules. This is a key component in solving Sudoku puzzles using CSP techniques, as it enables the identification and resolution of constraint violations.

By extending the `CSP` class, `SudokuCSP` can utilize the broader CSP-solving mechanisms (like backtracking, constraint propagation, and heuristics) while applying these specifically tailored constraint checks to ensure any proposed solution adheres to the Sudoku rules.

In [47]:
class SudokuCSP(CSP):
    def conflicts(self, i1, j1, x, i2, j2, y):
        k1 = i1 // 3 * 3 + j1 // 3
        k2 = i2 // 3 * 3 + j2 // 3
        return x == y and ( i1 == i2 or j1 == j2 or k1 == k2 )
    

## Class Definition: SudokuSolver

The `__addEdge__` method is intended to construct adjacency lists that define the constraints between different cells in the Sudoku board. It takes the row (`i`) and column (`j`) of a cell, along with an existing adjacency list (adjList), and updates the adjacency list to include all cells that are in the same row, column, or 3x3 subgrid as the given cell, except the cell itself. It calculates `k`, the index of the 3x3 subgrid to which the cell belongs, and iterates through all possible row and column indices (0 to 8) to add constraints. The method ensures that each cell is related to others that must have a different value according to Sudoku rules.

The `buildCspProblem` method initializes the CSP framework for solving a Sudoku puzzle. It creates an adjacency list (adjList), a list of variables (variables), a list of assigned variables (assigned), and domains (domains) for each cell. It uses `__addEdge__` to populate adjList with constraints based on the rules of Sudoku. For each cell, if the cell is empty (denoted by `.`), its domain is set to all possible values (1 through 9, adjusted to 0 through 8 to fit Python's 0-indexed nature). If a cell is already filled with a number, its domain is set to that specific number, and it's added to the list of assigned variables. The method returns an instance of SudokuCSP, initialized with the variables, adjacency list, and domains, along with a list of already assigned variables.

The `solveSudoku` method is intended to be the main function that takes a Sudoku board as input and solves it. The board is represented as a list of lists of strings, where each string can be a digit ('1' to '9') or '.' for empty cells.
The actual solving logic (CSP solving technique, backtracking, etc.) would be implemented within this method or called from this method. The placeholder pass indicates that the solving functionality is to be implemented.




In [4]:
class SudokuSolver(object):
    def __addEdge__(self, i, j, adjList):
        k = i // 3 * 3 + j // 3
        for num in range(9):
            if num != i:
                adjList[(i, j)].add((num, j))
            if num != j:
                adjList[(i, j)].add((i, num))
            row = num//3 + k//3 * 3
            col = num%3 + k%3 * 3
            if row != i or col != j:
                adjList[(i, j)].add((row, col))

    def buildCspProblem(self, board):
        adjList = defaultdict(set)
        # Build graph (contraints)
        for i in range(9):
            for j in range(9):
                self.__addEdge__(i, j, adjList)
        # Set domains
        variables = []
        assigned = []
        domains = {}
        for i in range(9):
            for j in range(9):
                if board[i][j] == '.':
                    domains[(i, j)] = set(range(9))
                    variables.append((i, j))
                else:
                    domains[(i, j)] = set([int(board[i][j]) - 1])
                    assigned.append((i, j))
        return SudokuCSP(variables, adjList, domains), assigned

    def solveSudoku(self, board):
        """
        :type board: List[List[str]]
        :rtype: void Do not return anything, modify board in-place instead.
        """
        pass

## Class Definition: BacktrackSudokuSolver

The `BacktrackSudokuSolver` class extends SudokuSolver to solve Sudoku puzzles using a backtracking algorithm.

`heng`, `zong`, and `gezi` are lists of sets, each corresponding to rows, columns, and 3x3 squares (grids) of the Sudoku board, respectively. These sets track the numbers already used in each row, column, and grid. `blank` is a list to store the positions `(i, j)` of all the empty cells `('.')` in the Sudoku board.The class iterates over the entire 9x9 Sudoku board. For each cell, if it contains a number (not `'.'`), the number (adjusted by -1 for 0-indexing) is added to the corresponding row, column, and grid sets. This step initializes the tracking structures with the given numbers in the puzzle. If the cell is empty (`'.'`), its position is added to blank for further processing.

The `backtrack` Function:
- A recursive function designed to try filling all empty cells with valid numbers.
- `start` is the index in blank to start the backtracking from.
- For each empty cell, identified by its position `(i, j)` from `blank`, the function tries numbers 0 through 8 (which correspond to Sudoku numbers 1 through 9).
- It checks if the current number is not already used in the same row (`heng[i]`), column (`zong[j]`), or grid (`gezi[k]`). If not, it places the number in the cell and recursively calls itself to try filling the next empty cell.
- If placing a number leads to a dead end (no valid number can be placed in the next empty cell(s)), it removes the number from the cell and the tracking sets, and tries the next number.
- The recursion base case is when start exceeds the number of empty cells, meaning the board is successfully filled. It returns True to signal success.


### Solving the Sudoku:
The `solveSudoku` method initializes the tracking structures and calls `backtrack(0)` to start solving the puzzle from the first empty cell.
Since the solution modifies the `board` in place, no return value is necessary. The return None at the end is optional since Python functions return `None` by default if no return statement is executed.



In [5]:
class BacktrackSudokuSolver(SudokuSolver):
    def solveSudoku(self, board):
        """
        :type board: List[List[str]]
        :rtype: void Do not return anything, modify board in-place instead.
        """
        heng = [set() for i in range(9)]
        zong = [set() for i in range(9)]
        gezi = [set() for i in range(9)]
        blank = []
        for i in range(9):
            for j in range(9):
                if board[i][j] != '.':
                    k = i // 3 * 3 + j // 3
                    num = ord(board[i][j]) - ord('0') - 1
                    heng[i].add(num); zong[j].add(num); gezi[k].add(num)
                else:
                    blank.append((i, j))

        def backtrack(start):
            if start >= len(blank):
                return True
            i, j = blank[start]
            k = i // 3 * 3 + j // 3
            for num in range(9):
                if num not in heng[i] and num not in zong[j] and\
                        num not in gezi[k]:
                    board[i][j] = chr(ord('0') + num + 1)
                    heng[i].add(num); zong[j].add(num); gezi[k].add(num)
                    if backtrack(start + 1):
                        return True
                    heng[i].remove(num); zong[j].remove(num); gezi[k].remove(num)
                    board[i][j] = '.'
            return False

        backtrack(0)
        return None
    
    

## AC-3

This code snippet implements the AC-3 (Arc Consistency Algorithm 3) for enforcing arc consistency in Constraint Satisfaction Problems (CSPs). The purpose of AC-3 is to reduce the search space by pruning values from variables' domains that are inconsistent with the constraints, thus potentially simplifying the CSP to a point where it can be solved more easily, or determining that no solution exists if a domain becomes empty.

`remove_inconsistent_values` Function:
- Purpose: Checks and removes values from the domain of a variable `Xt` that are inconsistent with another variable `Xh`.
- Process:
    - Iterates through each possible value `x` in the domain of `Xt`. For each `x`, it checks if there exists at least one value `y` in the domain of `Xh` such that `Xt=x` does not conflict with `Xh=y` based on the constraints defined in `csp.conflicts`.
    - If `x` conflicts with every possible `y`, `x` is removed from `Xt`'s domain, and this removal is recorded in removals.
    - Returns True if any value was removed, indicating the domain of `Xt` was revised.
    
    
`AC3` Function:
- Purpose: Establishes arc consistency for the entire CSP.
- Initialization:
    - If no queue of arcs is provided, it initializes the queue with all arcs in the CSP, represented as pairs of variables (Xt, Xh) where Xt and Xh have a constraint between them.
- Process:
    - Continuously processes each arc (Xt, Xh) in the queue. For each arc, it calls remove_inconsistent_values to potentially prune the domain of Xt.
    - If any value was removed (domain revised), it checks:
        - If `Xt`'s domain is empty, returns False, indicating no consistent assignment is possible.
        - Otherwise, it enqueues all arcs `(X, Xt)` for neighboring variables `X` of `Xt`, excluding the back arc to `Xh`, to recheck consistency due to the domain change in `Xt`.
- Outcome:
    - Returns `True` if arc consistency is achieved across all variables without emptying any domain, indicating a potential path towards a solution.
    - Returns `False` if any variable's domain is emptied, indicating no solution is possible under current constraints.
    
`makeArcQue` Function:
- Generates a queue of arcs given a subset of variables `Xs`. For each variable `Xh` in `Xs`, it pairs `Xh` with every variable `Xt` that is adjacent to `Xh` (as determined by the CSP's adjacency list).
- This function is useful for initializing the AC3 algorithm with a specific set of arcs, possibly focusing on parts of the CSP that were recently modified or are of particular interest.
- The resulting queue can be used to apply arc consistency checks to a targeted subset of the CSP's variables, which can be efficient in dynamic or incremental settings where only part of the CSP changes or requires reevaluation.
    
    

In [6]:
def remove_inconsistent_values(csp, Xt, Xh, removals):
    # Return True if we remove a value
    revised = False
    # If Xt=x conflicts with Xh=y for every possible y, eliminate Xt=x
    for x in csp.domains[Xt].copy():
        for y in csp.domains[Xh]:
            if not csp.conflicts(*Xt, x, *Xh, y):
                break
        else:
            csp.domains[Xt].remove(x)
            removals[Xt].add(x)
            revised = True
    return revised


def AC3(csp, queue=None, removals=defaultdict(set)):
    # Return False if there is no consistent assignment
    if queue is None:
        queue = [(Xt, X) for Xt in csp.adjList for X in csp.adjList[Xt]]
    # Queue of arcs of our concern
    while queue:
        # Xt --> Xh Delete from domain of Xt
        (Xt, Xh) = queue.pop()
        if remove_inconsistent_values(csp, Xt, Xh, removals):
            if not csp.domains[Xt]:
                return False
            # NOTE: Next two lines only for binary "!=" constraint
            elif len(csp.domains[Xt]) > 1:
                continue
            for X in csp.adjList[Xt]:
                if X != Xt:
                    queue.append((X, Xt))
    return True

def makeArcQue(csp, Xs):
    return [(Xt, Xh) for Xh in Xs for Xt in csp.adjList[Xh]]


## <font color='red'>Q1: Define a class: AC3SudokuSolver</font> 

Your code should define a class `AC3SudokuSolver` that extends `SudokuSolver` to solve Sudoku puzzles by integrating backtracking search with the AC3 algorithm for maintaining arc consistency. Here's an outline of how the solution process is implemented:

### Building the CSP Problem:
- The method `solveSudoku` starts by constructing the CSP model of the given Sudoku puzzle using the inherited `buildCspProblem` method, which sets up the variables, domains, and constraints (adjacency list) based on the initial board state.
- `assigned` captures the initially filled cells of the Sudoku board, which are treated as variables with a fixed domain (the given value).


### Applying AC3:
- The AC3 algorithm is applied to enforce arc consistency across the entire problem space, using `makeArcQue` to generate the queue of arcs from the initially assigned variables. This step aims to reduce the domains of variables by eliminating values that cannot participate in any valid solution, based on the current constraints.

### Identifying Uncertain Choices:
- The solver identifies cells (variables) with uncertain values—those whose domains contain more than one possible value—after the initial application of AC3. These cells are added to the `uncertain` list for further exploration through backtracking.

### Backtracking Search:
- The `backtrack` method is recursively called to explore different value assignments for the uncertain cells in an attempt to find a valid solution.
- For each uncertain cell `X`, the method iterates over all possible values `x` in its domain. For each `x`, it temporarily assigns `x` to `X`, updates the CSP model accordingly, and applies AC3 again to propagate the effects of this assignment.
- If AC3 indicates a consistent assignment is still possible (i.e., no domain is emptied), the search continues recursively with the next uncertain cell. If a solution is found (`retval` is `True`), the recursion unwinds successfully.
- If no valid assignment is found for `X`, the changes are undone (`csp.restore_domains`), and the method tries the next value of `X`.
- The uncertain cell `X` is added back to the list of uncertain cells if no solution is found with `X` being assigned any of its possible values.

### Updating the Original Board:
- Once a valid solution is found (all `uncertain` cells are successfully assigned), the method updates the original `board` with the solved values, converting them from 0-indexed to 1-indexed format (`csp.domains[(i, j)].pop() + 1`).


In [7]:
# Backtracking search with AC3 Maintaining Arc Consistency
class AC3SudokuSolver(SudokuSolver):
    def solveSudoku(self, board):

        # Build CSP problem
        csp, assigned = # put your code here
        
        # Enforce AC3 on initial assignments
        AC3(csp, makeArcQue(csp, assigned))
        
        # If there's still uncertain choices
        uncertain = []
        #put your code here
        
        
        
        
        # Search with backtracking
        self.backtrack(csp, uncertain)
        # Fill answer back to input table
        for i in range(9):
            for j in range(9):
                if board[i][j] == '.':
                    assert len(csp.domains[(i, j)]) == 1
                    board[i][j] = str( csp.domains[(i, j)].pop() + 1 )
    
    def backtrack(self, csp, uncertain):
        #put your code here
        
        
        
        
        


## <font color='red'>Q2: Define a class: AC3LCVSudokuSolver</font> 

Your code should outline an extension of the `AC3SudokuSolver`, creating a new class called `AC3LCVSudokuSolver`, which integrates the Least Constraining Value (LCV) heuristic into the backtracking search process. The LCV heuristic aims to choose the value that imposes the fewest constraints on the remaining unassigned variables, thereby potentially reducing the search space and leading to faster solutions. Let's break down the modifications and additions:

### `count_conflict` Method:

- This method calculates the number of conflicts a potential value `x` for a variable `Xi` would have with its adjacent variables according to the CSP's adjacency list. A conflict here means that the value `x` is present in the domain of an adjacent variable `X`.
- The `cnt` (count) represents how constraining the value `x` is if assigned to `Xi`. The fewer conflicts (`cnt`), the less constraining the value is considered.

### Modified `backtrack` Method:
- The `backtrack` method is overridden to incorporate the LCV heuristic into the value selection process for each uncertain variable.
- When selecting a value for variable `X` from its domain, the values are first sorted based on how many conflicts they generate with adjacent variables, as determined by the `count_conflict` method. This sorting orders the values from least to most constraining (Least Constraining Value).
- The search then proceeds by attempting to assign values to `X` in the order of least constraining to most constraining. This is done in the hope that choosing values that are less likely to constrain the domains of other variables will lead to fewer backtracks and a quicker solution.
- The rest of the `backtrack` method remains largely the same as in the base `AC3SudokuSolver`. It temporarily assigns a value `x` to `X`, applies AC3 to enforce arc consistency, and recursively continues the search. If no solution is found with `x`, the changes are undone, and the next least constraining value is tried.
- If all values for `X` are tried without success, `X` is added back to the list of uncertain variables, and the method backtracks.

### Outcome and Efficiency:
- By prioritizing the assignment of values that are least constraining to other variables, the `AC3LCVSudokuSolver` aims to navigate the search space more effectively than a straightforward backtracking approach. The LCV heuristic helps in minimizing the chance of future conflicts, thereby potentially reducing the number of backtracks needed to find a solution.
- The combination of AC3 for reducing the domain sizes through arc consistency and the LCV heuristic for intelligent value ordering makes this solver particularly adept at tackling complex Sudoku puzzles with efficiency.


In [8]:
# AC3 filtering with Least Constraining Value ordering
class AC3LCVSudokuSolver(AC3SudokuSolver):
    def count_conflict(self, csp, Xi, x):
        cnt = 0
        for X in csp.adjList[Xi]:
            if x in csp.domains[X]:
                cnt += 1
        return cnt

    def backtrack(self, csp, uncertain):
        #put your code here
        
        #Hint:Sort the values in domain in the order of LCV and loop in that order
        
        
        
        
        
        

## <font color='red'>Q3: Define a class: AC3MRVSudokuSolver</font> 

The `AC3MRVSudokuSolver` class should extend the `AC3SudokuSolver` to solve Sudoku puzzles by incorporating the Minimum Remaining Values (MRV) heuristic into the backtracking search process. MRV selects the variable with the fewest legal values left in its domain to assign next, potentially reducing the search space and increasing efficiency. Let's explore the functionalities added and modified in this solver:

### Modified `solveSudoku` Method:
- This method builds the CSP model from the given Sudoku board, applies AC3 to enforce arc consistency from the start, identifies uncertain cells (those with more than one possible value left), and then proceeds with backtracking search to find a solution.
- After finding a solution, it updates the original Sudoku board with the solved values.

### `popMinRandom` and `popMin` Methods:
- `popMinRandom` and `popMin` are utility methods designed to select and remove a variable from the `uncertain` list based on the MRV heuristic. While both methods aim to select the variable with the minimum number of legal values (minimum domain size), `popMinRandom` introduces randomness when there are multiple variables with the same smallest domain size by randomly choosing one of them. This could potentially provide a diversity of search paths in repeated runs. `popMin`, on the other hand, consistently chooses the first variable it finds with the minimum domain size.
- These methods rearrange the selected variable to the end of the array and then pop it, optimizing the removal process.

### Modified backtrack Method:
- Overrides the `backtrack` method from `AC3SudokuSolver` to integrate the MRV heuristic explicitly. Instead of popping the last item from `uncertain`, it uses `popMin` to select the next variable to assign based on MRV, aiming to reduce branching in the search tree.
- For each chosen variable `X`, it iterates over all possible values `x` in its domain, temporarily assigns `x` to `X`, and applies AC3 to propagate constraints and possibly reduce the domains of other variables further.
- If a consistent assignment is found (AC3 returns True), it recursively continues with the next variable. If the assignment leads to a dead-end, it restores the previous state and tries the next value.
- If all values for `X` are exhausted without success, `X` is added back to the list of uncertain variables for potential reassignment, and the method backtracks.

### Key Features:
- The integration of the MRV heuristic aims to choose the most constrained variable (the one with the fewest legal moves) as the next variable to assign, theoretically minimizing the number of decisions and backtracks needed.
- The MRV heuristic is particularly effective in constraint-dense problems like Sudoku, where early decisions significantly influence the complexity of remaining decisions.
- The `AC3MRVSudokuSolver` leverages both arc consistency (via AC3) to reduce domain sizes before and during search, and MRV to intelligently select the next variable to assign, aiming for a more efficient search process.

In [9]:
# AC3 filtering with Minimum Remaining Values ordering
class AC3MRVSudokuSolver(AC3SudokuSolver):
    def solveSudoku(self, board):
        """
        :type board: List[List[str]]
        :rtype: void Do not return anything, modify board in-place instead.
        """
        # Build CSP problem
        csp, assigned = self.buildCspProblem(board)
        # Enforce AC3 on initial assignments
        AC3(csp, makeArcQue(csp, assigned))
        # If there's still uncertain choices
        uncertain = []
        for i in range(9):
            for j in range(9):
                if len(csp.domains[(i, j)]) > 1:
                    uncertain.append((i, j))
        # Search with backtracking
        self.backtrack(csp, uncertain)
        # Fill answer back to input table
        for i in range(9):
            for j in range(9):
                if board[i][j] == '.':
                    assert len(csp.domains[(i, j)]) == 1
                    board[i][j] = str( csp.domains[(i, j)].pop() + 1 )

    def popMinRandom(self, array, key):
        minimum, indices = float("inf"), []
        for i in range(len(array)):
            if key(array[i]) < minimum:
                indices = [i]
                minimum = key(array[i])
            elif key(array[i]) == minimum:
                indices.append(i)
        idx = choice(indices)
        array[idx], array[-1] = array[-1], array[idx]
        return array.pop()

    def popMin(self, array, key):
        minimum, idx = float("inf"), 0
        for i in range(len(array)):
            if key(array[i]) < minimum:
                idx = i
                minimum = key(array[i])
        array[idx], array[-1] = array[-1], array[idx]
        return array.pop()

    def backtrack(self, csp, uncertain):
        #Put your code here
        
        
        
        
        
        


## <font color='red'>Q4: Define a class: AC3MRVLCVSudokuSolver</font> 

The `AC3MRVLCVSudokuSolver` class extends `AC3SudokuSolver` to incorporate both the MRV heuristic and the LCV heuristic into the Sudoku solving process. This combination aims to further optimize the backtracking search by intelligently selecting which variable to assign next (MRV) and in what order to try the possible values for that variable (LCV). Here's an in-depth look at how the class implements these strategies:

### Integrating MRV and LCV:
- MRV (Minimum Remaining Values): This heuristic selects the next variable to assign by choosing the one with the fewest legal (remaining) values in its domain. The idea is that variables with fewer options are more constrained and thus should be assigned earlier to reduce the branching factor in the search tree.
- LCV (Least Constraining Value): Once a variable is selected for assignment (using MRV), the LCV heuristic orders the values in its domain based on how many options they leave open for the adjacent variables (i.e., how constraining they are to the rest of the variables in the CSP). The value that leaves the most options available (least constraining) is tried first.

### Key Methods:
- `count_conflict`: Counts how many times a potential value for a variable appears in the domains of adjacent variables. It's used for the LCV heuristic to determine how constraining a value is.
- `popMin`: Extracts the variable with the minimum domain size from the list of uncertain variables, applying the MRV heuristic. It modifies the input list in place and returns the selected variable.
- `solveSudoku`:
    - Builds the CSP model from the Sudoku board.
    - Applies AC3 to enforce arc consistency, reducing the domains of variables before starting the search.
    - Identifies uncertain variables (those with domains larger than one value) and attempts to solve the CSP using backtracking.
    - If a solution is found, updates the original board with the solved values.
- `backtrack`:
    - Applies MRV to select the next variable to assign by using `popMin`.
    - Orders the values in the selected variable's domain according to LCV by sorting them based on the `count_conflict` measure.
    - Tries each value in order, setting the variable to that value and recursively attempting to solve the rest of the puzzle.
    - Uses AC3 after each assignment to maintain arc consistency, potentially reducing the domains of other variables based on the new assignment.
    - Restores previous domains if a value leads to a dead end, using `restore_domains`.
    - If no values lead to a solution for the current variable, adds the variable back to the list of uncertain variables and backtracks.

### Outcome:
By combining MRV and LCV, the `AC3MRVLCVSudokuSolver` aims to significantly improve the efficiency of solving Sudoku puzzles compared to using backtracking alone. The MRV heuristic helps reduce the size of the search tree by focusing on the most constrained variables first, while the LCV heuristic aims to minimize the impact of each assignment on the rest of the puzzle, ideally leading to fewer conflicts and backtracks. This dual heuristic approach, combined with arc consistency checks (AC3), makes for a powerful and efficient Sudoku solving strategy.

In [10]:
# AC3 filtering with Minimum Remaining Values and Least Constraining Value
# ordering
class AC3MRVLCVSudokuSolver(AC3SudokuSolver):
    def count_conflict(self, csp, Xi, x):
        cnt = 0
        for X in csp.adjList[Xi]:
            if x in csp.domains[X]:
                cnt += 1
        return cnt

    def popMin(self, array, key):
        minimum, idx = float("inf"), 0
        for i in range(len(array)):
            if key(array[i]) < minimum:
                idx = i
                minimum = key(array[i])
        array[idx], array[-1] = array[-1], array[idx]
        return array.pop()

    def solveSudoku(self, board):
        # Build CSP problem
        csp, assigned = self.buildCspProblem(board)
        # Enforce AC3 on initial assignments
        if not AC3(csp, makeArcQue(csp, assigned)):
            return False
        # If there's still uncertain choices
        uncertain = []
        for i in range(9):
            for j in range(9):
                if len(csp.domains[(i, j)]) > 1:
                    uncertain.append((i, j))
        # Search with backtracking
        if not self.backtrack(csp, uncertain):
            return False
        # Fill answer back to input table
        for i in range(9):
            for j in range(9):
                if board[i][j] == '.':
                    assert len(csp.domains[(i, j)]) == 1
                    board[i][j] = str( csp.domains[(i, j)].pop() + 1 )
        return True

    def backtrack(self, csp, uncertain):
        #put your code here
        
        
        
        
        
        
        

## Comparing Sudoku Solvers


In [52]:
solvers = [BacktrackSudokuSolver(), AC3SudokuSolver(),
            AC3LCVSudokuSolver(), AC3MRVSudokuSolver(), AC3MRVLCVSudokuSolver()]
names = ["Backtrack", "Backtrack AC3", "AC3 LCV", "AC3 MRV",
            "AC3 MRV LCV"]

def printBoard(board):
        return "\n".join(map(" ".join, board))
    
with open("stress_test.txt", "r") as fd:
            cases = list(map(eval, fd.read().splitlines()))
        
#for i in range(len(cases)):
for i in range(5):
    c = cases[i]
    print(f"problem {i}:")
    print(printBoard(c))
    for n, s in enumerate(solvers):
        s.solveSudoku(c)
        print(f"{names[n]} solution:")
        print(printBoard(c))


problem 0:
. 5 . . . 9 . . .
. . . . 6 8 3 1 .
. . . 1 . . . 9 .
. . . . . . . 6 .
. 7 . . 4 . . . .
. . . . 9 . . . .
. 4 . 8 5 2 . . .
. . . . . 6 . 4 .
7 . 8 . . . . . .
Backtrack solution:
1 5 2 4 3 9 6 7 8
4 9 7 5 6 8 3 1 2
3 8 6 1 2 7 4 9 5
2 1 3 7 8 5 9 6 4
5 7 9 6 4 1 2 8 3
8 6 4 2 9 3 1 5 7
6 4 1 8 5 2 7 3 9
9 2 5 3 7 6 8 4 1
7 3 8 9 1 4 5 2 6
Backtrack AC3 solution:
1 5 2 4 3 9 6 7 8
4 9 7 5 6 8 3 1 2
3 8 6 1 2 7 4 9 5
2 1 3 7 8 5 9 6 4
5 7 9 6 4 1 2 8 3
8 6 4 2 9 3 1 5 7
6 4 1 8 5 2 7 3 9
9 2 5 3 7 6 8 4 1
7 3 8 9 1 4 5 2 6
AC3 LCV solution:
1 5 2 4 3 9 6 7 8
4 9 7 5 6 8 3 1 2
3 8 6 1 2 7 4 9 5
2 1 3 7 8 5 9 6 4
5 7 9 6 4 1 2 8 3
8 6 4 2 9 3 1 5 7
6 4 1 8 5 2 7 3 9
9 2 5 3 7 6 8 4 1
7 3 8 9 1 4 5 2 6
AC3 MRV solution:
1 5 2 4 3 9 6 7 8
4 9 7 5 6 8 3 1 2
3 8 6 1 2 7 4 9 5
2 1 3 7 8 5 9 6 4
5 7 9 6 4 1 2 8 3
8 6 4 2 9 3 1 5 7
6 4 1 8 5 2 7 3 9
9 2 5 3 7 6 8 4 1
7 3 8 9 1 4 5 2 6
AC3 MRV LCV solution:
1 5 2 4 3 9 6 7 8
4 9 7 5 6 8 3 1 2
3 8 6 1 2 7 4 9 5
2 1 3 7 8 5 9 6 4
5 7 9